# 6.1 Audio Encoder — Design Experiment (Step 1 of 2)

**Kaggle edition** — satu notebook untuk semua `MEMBER` (1–4). Jalankan sel **Setup** dan **Cache**
sebelum sel eksperimen. Detail langkah ada di `notebooks/kaggle/README.md`.

**Tujuan:** memilih **audio encoder terbaik** untuk *text-guided automatic gain staging* pada
MUSDB18-HQ, dengan protokol yang sama untuk semua encoder.

---

## Kontrak fairness

| Aspek | Nilai |
| --- | --- |
| Text encoder | `indobenchmark/indobert-base-p1`, fine-tuned |
| Text hyperparameter | Ditune **global sekali** setelah audio winner dipilih |
| Audio hyperparameter | Ditune **per member** untuk encoder masing-masing |
| Head / arsitektur | M3 Structured Two-Branch |
| Loss | `MSE(g) + 0.1*MSE(L) + 0.1*MSE(r)` sesuai mode |
| Label | `compute_gain_targets` di `wadidaw/data/features.py` |
| CV | 5-fold song-level pada 80 lagu train |
| Validation | 20 lagu, tidak masuk CV |
| Test | 50 lagu, tidak dipakai sampai evaluasi final |
| Optimizer | AdamW |
| Seeds | tuning `42`; final `42, 123, 2026` |
| Metrik | `evaluate_model` (MAE/RMSE/band/success/adherence/per-stem) |

**Yang berubah hanya audio encoder:** `cnn_scratch`, `ast`, `mert`, `beats` — semuanya trainable.

## Hyperparameter yang ditune

Audio per member:
- `lr_audio` / `lr_scratch`
- `wd_audio`
- `dropout_audio`

Text global:
- `lr_text`
- `dropout_text`
- `text_hidden`

Text branch tidak ditune per member agar tetap menjadi variabel kontrol.


## Cara menjalankan di Kaggle

### Prasyarat sekali saja
1. Upload dataset audio `satriamanggala/musdb18hq` (31,2 GB).
2. Upload cache log-mel `data/cache/logmel` (593 MB) sebagai private dataset.
3. **Add Input** → dataset audio + dataset cache.
4. **Session options** → Accelerator `GPU T4 x2`, **Internet ON**.

### Stage 1 — Audio tuning per member
1. Set `MEMBER` di **Cell 1** (atau env `WADIDAW_MEMBER`).
2. Jalankan Setup → Cache → Cell 1 → Cell 2–8.
3. Set `RUN_AUDIO_TUNING = True` di **Cell 1**.
4. Jalankan **Cell 9**. Optuna menyimpan study di `member<N>/optuna/` dan hasil di
   `member<N>/best_audio_config_<encoder>.json`.

### Stage 2 — Text tuning global
1. Ketua menggabungkan hasil audio dan memilih encoder pemenang.
2. Hanya member pemilik encoder pemenang yang mengisi:
   ```python
   TEXT_TUNING_ENCODER = "ast"   # contoh
   TEXT_TUNING_MEMBER = 2        # contoh
   RUN_TEXT_TUNING = True
   ```
3. Jalankan **Cell 9**. Hasil: `outputs/encoder_selection/text/text_config.json`.

### Stage 3 — Final model
1. Member pemilik encoder pemenang mengisi:
   ```python
   FINAL_ENCODER = "ast"         # contoh
   RUN_FINAL = True
   ```
2. Jalankan **Cell 9**. Hasil: `final/final_<encoder>_seed*.json` dan checkpoint.

### Stage 4 — Text-only ablation (opsional)
Member 2 dapat menjalankan `RUN_TEXT_ONLY = True` untuk ablasi `text_only` 5-fold.

> Satu sesi Kaggle maksimal 12 jam. Gunakan `MAX_TRIALS_PER_SESSION` dan `TIME_BUDGET_MIN`
> untuk berhenti rapi dan melanjutkan di sesi berikutnya.


In [ ]:
# ----------------------------------------------------------------------------- Kaggle setup
import json
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/savitar393/wadidaw-studio.git"
BRANCH = "feat/audio-encoder-design-experiment-notebook"
REPO_DIR = Path("/kaggle/working/wadidaw-studio")
INPUT_ROOT = Path("/kaggle/input")


def run(cmd: str, check: bool = True) -> None:
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=check)


def has_internet(host: str = "github.com") -> bool:
    try:
        socket.gethostbyname(host)
        return True
    except OSError:
        return False


def find_uploaded_repo() -> Path | None:
    for pat in ("*", "*/*", "*/*/*"):
        for candidate in INPUT_ROOT.glob(pat):
            try:
                if (candidate / "wadidaw").is_dir() and (candidate / "data" / "splits").is_dir():
                    return candidate
            except OSError:
                continue
    return None


if not (REPO_DIR / "wadidaw").is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    if has_internet("github.com"):
        run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    else:
        uploaded_repo = find_uploaded_repo()
        if uploaded_repo is not None:
            print("Internet OFF -> menyalin repo dari dataset:", uploaded_repo)
            shutil.copytree(uploaded_repo, REPO_DIR)
        else:
            raise RuntimeError(
                "Tidak bisa mengakses github.com (Internet Kaggle masih OFF). "
                "Buka panel kanan -> Session options -> nyalakan Internet ON."
            )
else:
    print("repo sudah ada:", REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("cwd:", Path.cwd())

# transformers untuk IndoBERT/AST/MERT, soundfile untuk baca WAV.
if has_internet("pypi.org"):
    run(f"{sys.executable} -m pip install -q 'transformers>=4.40' soundfile scipy 'optuna>=3.6'", check=False)
else:
    print("peringatan: Internet OFF, skip pip install (download IndoBERT/MERT juga butuh Internet ON)")


def looks_like_musdb(path: Path) -> bool:
    try:
        return (
            (path / "train").is_dir()
            and (path / "test").is_dir()
            and next((path / "train").glob("*/mixture.wav"), None) is not None
        )
    except OSError:
        return False


def find_musdb_root() -> Path | None:
    patterns = ("*", "*/*", "*/*/*", "*/*/*/*")
    candidates = [INPUT_ROOT, *(p for pat in patterns for p in INPUT_ROOT.glob(pat))]
    for candidate in candidates:
        if looks_like_musdb(candidate):
            return candidate
    return None


MUSDB_ROOT_KAGGLE = find_musdb_root()
if MUSDB_ROOT_KAGGLE is None:
    zips = sorted(INPUT_ROOT.glob("**/*.zip"))
    if zips:
        dest = Path("/kaggle/tmp/musdb18hq")
        dest.mkdir(parents=True, exist_ok=True)
        run(f"unzip -q -o '{zips[0]}' -d '{dest}'")
        MUSDB_ROOT_KAGGLE = find_musdb_root() or next(
            (p for p in [dest, *dest.glob("*"), *dest.glob("*/*")] if looks_like_musdb(p)), None
        )

if MUSDB_ROOT_KAGGLE is None:
    print("Isi /kaggle/input (debug):")
    for pat in ("*", "*/*", "*/*/*"):
        for p in sorted(INPUT_ROOT.glob(pat)):
            if p.is_dir():
                print(" -", p)
    raise AssertionError("MUSDB18-HQ tidak ditemukan di /kaggle/input (cek Add Input)")

print("MUSDB18-HQ:", MUSDB_ROOT_KAGGLE)

# Symlink supaya kode yang memakai MUSDB_ROOT bawaan repo tetap benar.
link = REPO_DIR / "data" / "raw" / "musdb18hq"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.exists():
    print("path MUSDB lokal sudah ada:", link)
else:
    link.symlink_to(MUSDB_ROOT_KAGGLE, target_is_directory=True)
    print("symlink:", link, "->", MUSDB_ROOT_KAGGLE)

# Monkeypatch default modul agar import MUSDB_ROOT di Cell 1 memakai path Kaggle.
import wadidaw.data.musdb as _musdb_module

_musdb_module.MUSDB_ROOT = MUSDB_ROOT_KAGGLE

split_manifest = json.loads((REPO_DIR / "data" / "splits" / "musdb18hq_v1.json").read_text())
songs = [member for split in split_manifest["splits"].values() for member in split]
missing_songs = [
    member["song"]
    for member in songs
    if not (MUSDB_ROOT_KAGGLE / str(member["relative_dir"])).is_dir()
]
print(f"split songs: {len(songs)} | folder tidak ditemukan: {len(missing_songs)}")
if missing_songs:
    print("contoh hilang:", missing_songs[:5])
    print("contoh folder dataset:", [p.name for p in (MUSDB_ROOT_KAGGLE / "train").iterdir()][:5])

import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# ----------------------------------------------------------------------------- Kaggle cache (log-mel + waveform)
from wadidaw.data.dataset import SAMPLES_ROOT

LOGMEL_CACHE = REPO_DIR / "data" / "cache" / "logmel"
WAVEFORM_CACHE_DIR = REPO_DIR / "data" / "cache" / "waveform16k"
BUILD_LOGMEL_IF_MISSING = True   # False -> gagal cepat kalau cache belum di-upload


def has_npy_cache(path: Path) -> bool:
    try:
        return (path / "train").is_dir() and next((path / "train").glob("*.npy"), None) is not None
    except OSError:
        return False


def find_uploaded_cache(name: str) -> Path | None:
    patterns = ("*", "*/*", "*/*/*", "*/*/*/*")
    for pat in patterns:
        for candidate in INPUT_ROOT.glob(f"{pat}/{name}"):
            if has_npy_cache(candidate):
                return candidate
    if name == "logmel":  # dataset di-upload tanpa folder pembungkus
        for pat in patterns:
            for candidate in INPUT_ROOT.glob(pat):
                if has_npy_cache(candidate):
                    return candidate
    return None


def link_cache(target: Path, source: Path) -> None:
    if target.is_symlink():
        print("cache symlink sudah ada:", target)
        return
    if target.exists():
        # Folder kosong sisa build yang gagal boleh diganti symlink.
        if any(target.iterdir()):
            print("cache sudah ada:", target)
            return
        target.rmdir()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.symlink_to(source, target_is_directory=True)
    print("symlink cache:", target, "->", source)


uploaded_logmel = find_uploaded_cache("logmel")
if uploaded_logmel is not None:
    link_cache(LOGMEL_CACHE, uploaded_logmel)
elif has_npy_cache(LOGMEL_CACHE):
    print("log-mel cache sudah ada:", LOGMEL_CACHE)
elif BUILD_LOGMEL_IF_MISSING:
    print("log-mel cache belum ada -> membangun dari WAV (30-60 menit) ...")
    run(
        f"{sys.executable} -m scripts.build_dataset "
        f"--musdb-root '{MUSDB_ROOT_KAGGLE}' "
        f"--manifest '{REPO_DIR / 'data' / 'splits' / 'musdb18hq_v1.json'}' "
        f"--cache-root '{LOGMEL_CACHE}' "
        "--samples-root /kaggle/tmp/prompts_kaggle "
        "--workers 2"
    )
else:
    raise FileNotFoundError(
        "log-mel cache tidak ditemukan. Upload data/cache/logmel sebagai Kaggle dataset "
        "atau set BUILD_LOGMEL_IF_MISSING=True."
    )

uploaded_waveform = find_uploaded_cache("waveform16k")
if uploaded_waveform is not None:
    link_cache(WAVEFORM_CACHE_DIR, uploaded_waveform)

missing_audio = []
for manifest_name in (
    "train_samples.json",
    "validation_samples.json",
    "test_human_samples.json",
    "test_synthetic_samples.json",
):
    manifest_path = SAMPLES_ROOT / manifest_name
    if not manifest_path.is_file():
        continue
    for record in json.loads(manifest_path.read_text()):
        if not (REPO_DIR / str(record["audio_path"])).is_file():
            missing_audio.append(record["audio_path"])

print(f"manifest audio files missing: {len(missing_audio)}")
if missing_audio:
    print("contoh:", missing_audio[:3])
    raise FileNotFoundError("cache log-mel tidak lengkap; cek dataset cache yang di-attach")
print("cache siap:", LOGMEL_CACHE)


In [ ]:
# ----------------------------------------------------------------------------- Cell 1: bootstrap + config
import copy
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "wadidaw").is_dir():
            return candidate
    return start


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from wadidaw.data.dataset import SAMPLES_ROOT, MusdbMultimodalDataset
from wadidaw.data.features import ACTIVE_THRESHOLD_DB, resample_to_16k_mono
from wadidaw.data.musdb import MUSDB_ROOT
from wadidaw.data.prompts import STEM_NAMES
from wadidaw.models.structured import compute_multimodal_loss
from wadidaw.training.evaluate import evaluate_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WAVEFORM_CACHE = ROOT / "data" / "cache" / "waveform16k"
OUT_DIR = ROOT / "outputs" / "encoder_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "train_manifest": str(SAMPLES_ROOT / "train_samples.json"),
    "val_manifest": str(SAMPLES_ROOT / "validation_samples.json"),
    "test_human_manifest": str(SAMPLES_ROOT / "test_human_samples.json"),
    "test_synthetic_manifest": str(SAMPLES_ROOT / "test_synthetic_samples.json"),
    "text_model": "indobenchmark/indobert-base-p1",
    "d_model": 128,
    "text_hidden": 128,
    "dropout_text": 0.15,
    "dropout_audio": 0.10,
    "batch_size": 4,          # dikunci 4 di semua mesin
    "grad_accum_steps": 4,    # dikunci 4 di semua mesin -> effective batch 16
    "epochs": 30,
    "patience": 5,
    "lr_head": 1e-3,
    "lr_audio": 1e-4,
    "lr_scratch": 1e-3,
    "lr_text": 1e-4,
    "wd_audio": 1e-4,
    "wd_text": 1e-4,
    "wd_head": 1e-4,
    "eta_min": 1e-6,
    "grad_clip": 1.0,
    "seeds": [42, 123, 2026],
    "use_amp": True,
    "encoders": ["cnn_scratch", "ast", "mert", "beats"],
    "modes": ["bimodal", "audio_only", "text_only"],
    "waveform_rate": 16000,
    "n_folds": 5,
    "cv_seed": 2026,
    "tuning_seed": 42,
    "n_trials": 10,
}
# Auxiliary-loss weights per ablation mode (lambda_loudness, lambda_balance).
AUX_LAMBDA = {"bimodal": (0.1, 0.1), "audio_only": (0.1, 0.0), "text_only": (0.0, 0.1)}

# --- Pembagian encoder ke 4 anggota -----------------------------------------
MEMBER = int(os.environ.get("WADIDAW_MEMBER", "1"))
SHARDS = {
    1: {"encoders": ["cnn_scratch"], "modes": ["bimodal", "audio_only"]},                 # iGPU Arc / CPU, 16 GB
    2: {"encoders": ["ast"],          "modes": ["bimodal", "audio_only", "text_only"]},   # RTX 3050 4 GB, 24 GB
    3: {"encoders": ["mert"],         "modes": ["bimodal", "audio_only"]},                 # RTX 4050 6 GB, 32 GB
    4: {"encoders": ["beats"],        "modes": ["bimodal", "audio_only"]},                 # RTX 5050 8 GB, 16 GB
}
effective_batch = 16
assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus 16"
assert MEMBER in SHARDS, f"MEMBER harus 1..4, bukan {MEMBER}"

RUN_DIR = OUT_DIR / f"member{MEMBER}"
FOLD_FILE = ROOT / "data" / "splits" / "cv5_train_songs.json"
FOLD_DIR = OUT_DIR / "folds"
TEXT_DIR = OUT_DIR / "text"
FINAL_DIR = OUT_DIR / "final"
CV_DIR = RUN_DIR / "cv"
STUDY_DIR = RUN_DIR / "optuna"
for _d in (OUT_DIR, RUN_DIR, FOLD_DIR, TEXT_DIR, FINAL_DIR, CV_DIR, STUDY_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Stage flags -------------------------------------------------------------
# Set True hanya untuk stage yang ingin dijalankan.
RUN_AUDIO_TUNING = False
RUN_TEXT_TUNING = False
RUN_FINAL = False
RUN_TEXT_ONLY = False

# Setelah leaderboard audio, isi dua variabel ini.
# Contoh: TEXT_TUNING_ENCODER = "ast"; TEXT_TUNING_MEMBER = 2
TEXT_TUNING_ENCODER = None
TEXT_TUNING_MEMBER = None

# Setelah text tuning selesai, isi encoder pemenang.
# Contoh: FINAL_ENCODER = "ast"
FINAL_ENCODER = None

# --- Search spaces -----------------------------------------------------------
AUDIO_SEARCH_SPACE = {
    "pretrained": {
        "lr_audio": [5e-5, 1e-4, 2e-4],
        "wd_audio": [1e-4, 1e-3],
        "dropout_audio": [0.0, 0.1, 0.2],
    },
    "scratch": {
        "lr_scratch": [5e-4, 1e-3, 2e-3],
        "wd_audio": [1e-4, 1e-3],
        "dropout_audio": [0.0, 0.1, 0.2],
    },
}
TEXT_SEARCH_SPACE = {
    "lr_text": [5e-5, 1e-4, 2e-4],
    "dropout_text": [0.10, 0.15, 0.20],
    "text_hidden": [64, 128, 256],
}

SAVE_FINAL_CHECKPOINT = True
MAX_TRIALS_PER_SESSION = 0   # 0 = tanpa batas
TIME_BUDGET_MIN = 660
NUM_WORKERS = 2 if Path("/kaggle/working").exists() else 0

print("device:", DEVICE)
print("repo root:", ROOT)
print("waveform cache:", WAVEFORM_CACHE)
print(f"member {MEMBER}: encoders={SHARDS[MEMBER]['encoders']} modes={SHARDS[MEMBER]['modes']} "
      f"(effective batch {effective_batch})")
print(f"stage flags: audio={RUN_AUDIO_TUNING} text={RUN_TEXT_TUNING} final={RUN_FINAL} text_only={RUN_TEXT_ONLY}")
print(f"tuning: n_trials={CONFIG['n_trials']} tuning_seed={CONFIG['tuning_seed']} final_seeds={CONFIG['seeds']}")


## Pembagian kerja 4 anggota

| Anggota | Encoder | Hyperparameter audio yang ditune |
| --- | --- | --- |
| 1 | `cnn_scratch` | `lr_scratch`, `wd_audio`, `dropout_audio` |
| 2 | `ast` | `lr_audio`, `wd_audio`, `dropout_audio` |
| 3 | `mert` | `lr_audio`, `wd_audio`, `dropout_audio` |
| 4 | `beats` | `lr_audio`, `wd_audio`, `dropout_audio` |

Text hyperparameter ditune **global sekali** oleh member pemilik encoder pemenang:
`lr_text`, `dropout_text`, `text_hidden`.

Batch dikunci `4 × 4 = effective 16` di semua mesin.

### Alur
1. Semua member menjalankan **audio tuning** 5-fold.
2. Ketua menjalankan **leaderboard** dan memilih encoder pemenang.
3. Member pemenang menjalankan **text tuning global**.
4. Member pemenang menjalankan **final model** 3 seed.
5. Ketua menggabungkan `best_audio_config_*.json`, `text/text_config.json`, dan
   `final/final_summary_*.json`.

### Merge
Setelah semua member push:
```bash
git pull
```
lalu jalankan **Cell 10 (Leaderboard)** untuk menggabungkan hasil dari
`outputs/encoder_selection/member*`.

> Anggota 2 (4 GB): AST fine-tune `batch 4` bisa OOM. Fallback yang tetap adil adalah
> turunkan `batch_size` dan naikkan `grad_accum_steps` agar hasil kali tetap 16 (mis. `2×8`).


## 1. Cache waveform 16 kHz

AST/BEATs memakai 16 kHz, MERT-v1 aslinya 24 kHz. Kita cache **satu** waveform 16 kHz mono
(reuse `resample_to_16k_mono`, konsisten dengan cache log-mel) dan memberi MERT input 16 kHz
sebagai *documented domain shift*; upgrade ke 24 kHz jika MERT menang.

Setiap excerpt di-cache sebagai `(4, 128000)` float16 (stem urut `STEM_NAMES`), **raw tanpa g0** —
perturbasi g0 diterapkan saat load, sama seperti cache log-mel.

In [2]:
# ----------------------------------------------------------------------------- Cell 2: waveform cache (idempotent)


def excerpt_id_from_audio_path(audio_path: str) -> tuple[str, str]:
    p = Path(audio_path)
    return p.stem, p.parent.name  # (excerpt_id, split_tag)


def waveform_cache_path(audio_path: str) -> Path:
    ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
    return WAVEFORM_CACHE / split_tag / f"{ex_id}.npy"


def musdb_subset_for_split(split_tag: str) -> str:
    return "train" if split_tag in ("train", "validation") else "test"


def build_waveform_cache(manifests, overwrite: bool = False) -> None:
    """Cache 16 kHz mono (4, 128000) float16 waveforms for every excerpt in the manifests."""
    unique: dict[str, dict] = {}
    for manifest in manifests:
        if not Path(manifest).is_file():
            continue
        for record in MusdbMultimodalDataset(manifest).samples:
            unique.setdefault(str(record["audio_path"]), record)

    built = 0
    for audio_path, record in unique.items():
        npy_path = waveform_cache_path(audio_path)
        if npy_path.exists() and not overwrite:
            continue
        _ex_id, split_tag = excerpt_id_from_audio_path(audio_path)
        song_dir = MUSDB_ROOT / musdb_subset_for_split(split_tag) / str(record["song"])
        start = int(round(float(record["excerpt_start_sec"]) * 44100))
        frames = int(round(float(record["duration_sec"]) * 44100))
        stems = []
        for stem in STEM_NAMES:
            audio, sr = sf.read(song_dir / f"{stem}.wav", start=start, frames=frames, dtype="float32")
            stems.append(resample_to_16k_mono(audio, orig_sr=sr))
        npy_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(npy_path, np.stack(stems, axis=0).astype(np.float16))
        built += 1
    print(f"waveform cache: built {built}, total excerpts {len(unique)}")


CACHE_MANIFESTS = [CONFIG[k] for k in ("train_manifest", "val_manifest", "test_human_manifest", "test_synthetic_manifest")]
if not WAVEFORM_CACHE.exists():
    build_waveform_cache(CACHE_MANIFESTS)
else:
    print("waveform cache already present:", WAVEFORM_CACHE)

waveform cache: built 1500, total excerpts 1500


## 2. Dataset

`EncoderDataset` membungkus `MusdbMultimodalDataset` (log-mel `(4, 64, 800)` + label) dan menambahkan
`waveform` `(4, 128000)` dengan perturbasi `g0` yang sama. `cache_in_memory=False` untuk menekan RAM.

In [3]:
# ----------------------------------------------------------------------------- Cell 3: dataset


class EncoderDataset(Dataset):
    """MusdbMultimodalDataset + raw 16 kHz waveform (same g0 perturbation as the log-mel input)."""

    def __init__(self, manifest, root_dir=ROOT, cache_in_memory: bool = False):
        self.root_dir = Path(root_dir)
        self.base = MusdbMultimodalDataset(manifest, root_dir=root_dir, cache_in_memory=cache_in_memory)

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, index: int) -> dict:
        item = self.base[index]
        record = self.base.samples[index]
        wf_path = waveform_cache_path(str(record["audio_path"]))
        if not wf_path.is_file():
            raise FileNotFoundError(f"Missing waveform cache {wf_path}. Run Cell 2 first.")
        waveform = np.load(wf_path).astype(np.float32)
        g0 = np.asarray(record["g0_db"], dtype=np.float32)
        mask = np.asarray(record.get("active_mask", [True] * 4), dtype=bool)
        eff = np.where(mask, g0, 0.0).astype(np.float32)
        item["waveform"] = torch.from_numpy(waveform * np.power(10.0, eff / 20.0)[:, None])
        return item


_ds = EncoderDataset(CONFIG["val_manifest"])
_s = _ds[0]
print("keys:", [k for k in _s if k in ("audio", "waveform", "gains_db", "loudness_db", "balance_target", "active_mask")])
print("audio", tuple(_s["audio"].shape), "| waveform", tuple(_s["waveform"].shape), "| gains", tuple(_s["gains_db"].shape))

keys: ['audio', 'loudness_db', 'balance_target', 'gains_db', 'active_mask', 'waveform']
audio (4, 64, 800) | waveform (4, 128000) | gains (4,)


## 2b. 5-fold song-level (80 lagu train)

5-fold hanya memakai **80 lagu train**. Validation 20 lagu dan test 50 lagu tidak masuk CV.

Setiap fold: 64 lagu train / 16 lagu validation. Fold dibuat sekali, disimpan ke
`data/splits/cv5_train_songs.json`, dan dipakai oleh semua member.


In [ ]:
# ----------------------------------------------------------------------------- Cell 3b: 5-fold song-level


def _load_manifest(path):
    data = json.loads(Path(path).read_text())
    if isinstance(data, dict) and "samples" in data:
        return list(data["samples"])
    if isinstance(data, list):
        return data
    raise ValueError(f"Unrecognized manifest format: {path}")


def build_cv_folds(force: bool = False) -> dict:
    """Create deterministic song-level 5-fold assignment for the 80 train songs."""
    if FOLD_FILE.is_file() and not force:
        return json.loads(FOLD_FILE.read_text())

    records = _load_manifest(CONFIG["train_manifest"])
    songs = sorted({str(r["song"]) for r in records})
    n_folds = int(CONFIG["n_folds"])
    if len(songs) % n_folds != 0:
        raise ValueError(f"Jumlah lagu train ({len(songs)}) tidak habis dibagi {n_folds} fold.")
    if len(songs) != 80:
        raise ValueError(f"Protokol mengharapkan 80 lagu train, ditemukan {len(songs)}.")

    rng = random.Random(int(CONFIG["cv_seed"]))
    rng.shuffle(songs)

    per_fold = len(songs) // n_folds
    folds = []
    for k in range(n_folds):
        val_songs = songs[k * per_fold : (k + 1) * per_fold]
        val_set = set(val_songs)
        train_songs = [s for s in songs if s not in val_set]
        folds.append({
            "fold": k,
            "val_songs": val_songs,
            "train_songs": train_songs,
        })

    payload = {
        "seed": int(CONFIG["cv_seed"]),
        "n_folds": n_folds,
        "source": "train_only",
        "folds": folds,
    }
    FOLD_FILE.parent.mkdir(parents=True, exist_ok=True)
    FOLD_FILE.write_text(json.dumps(payload, indent=2))
    return payload


def write_fold_manifests(fold_payload: dict) -> Path:
    records = _load_manifest(CONFIG["train_manifest"])
    FOLD_DIR.mkdir(parents=True, exist_ok=True)
    for fold in fold_payload["folds"]:
        k = int(fold["fold"])
        val_songs = set(fold["val_songs"])
        train_records = [r for r in records if str(r["song"]) not in val_songs]
        val_records = [r for r in records if str(r["song"]) in val_songs]
        (FOLD_DIR / f"fold{k}_train.json").write_text(json.dumps(train_records, indent=2))
        (FOLD_DIR / f"fold{k}_val.json").write_text(json.dumps(val_records, indent=2))
    return FOLD_DIR


FOLD_PAYLOAD = build_cv_folds()
write_fold_manifests(FOLD_PAYLOAD)
FOLDS = FOLD_PAYLOAD["folds"]
print("fold file:", FOLD_FILE)
for _fold in FOLDS:
    print(f"  fold {_fold['fold']}: train {len(_fold['train_songs'])} lagu | val {len(_fold['val_songs'])} lagu")


## 3. Fitur level eksplisit

Dihitung **dari input** encoder (bukan dari label), jadi tidak ada leakage:
`N_LEVEL_FEATS = 7` → RMS dBFS, band-low, band-mid, band-high (dB), peak dB, crest factor, active mask.

In [4]:
# ----------------------------------------------------------------------------- Cell 4: level features
N_LEVEL_FEATS = 7  # rms, band_low, band_mid, band_high, peak, crest, active


def _assemble_level(rms_db: torch.Tensor, bands: torch.Tensor, peak_db: torch.Tensor) -> torch.Tensor:
    crest = peak_db - rms_db
    active = (rms_db > ACTIVE_THRESHOLD_DB).to(rms_db.dtype)
    stacked = torch.stack(
        [rms_db, bands[..., 0], bands[..., 1], bands[..., 2], peak_db, crest, active * 30.0], dim=-1
    )
    return stacked / 30.0  # rough fixed scaling; the level MLP handles the rest


def mel_level_features(audio_norm: torch.Tensor) -> torch.Tensor:
    """Level features from normalized dBFS log-mel (B, 4, 64, T). dB = x * 50 - 30."""
    db = audio_norm * 50.0 - 30.0
    power = torch.pow(10.0, db / 10.0)
    rms_db = 10.0 * torch.log10(power.mean(dim=(2, 3)).clamp_min(1e-12))
    peak_db = db.amax(dim=(2, 3))
    bands = 10.0 * torch.log10(
        torch.stack(
            [power[:, :, :16].mean(dim=(2, 3)), power[:, :, 16:40].mean(dim=(2, 3)), power[:, :, 40:].mean(dim=(2, 3))],
            dim=-1,
        ).clamp_min(1e-12)
    )
    return _assemble_level(rms_db, bands, peak_db)


def waveform_level_features(waveform: torch.Tensor, sample_rate: int = 16000) -> torch.Tensor:
    """Level features from raw waveform (B, 4, S)."""
    x = waveform.float()
    rms_db = 20.0 * torch.log10(x.pow(2).mean(dim=-1).clamp_min(1e-12).sqrt())
    peak_db = 20.0 * torch.log10(x.abs().amax(dim=-1).clamp_min(1e-12))
    spec = torch.fft.rfft(x, dim=-1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    freqs = torch.fft.rfftfreq(x.shape[-1], d=1.0 / sample_rate).to(x.device)
    band_db = []
    for lo, hi in ((0, 250), (250, 2000), (2000, sample_rate // 2)):
        sel = (freqs >= lo) & (freqs < hi)
        band_db.append(10.0 * torch.log10(power[..., sel].mean(dim=-1).clamp_min(1e-12)))
    return _assemble_level(rms_db, torch.stack(band_db, dim=-1), peak_db)


def compute_level_features(x: torch.Tensor, input_key: str) -> torch.Tensor:
    """All encoders use the same waveform-level features.

    The encoder content path may differ, but level features are always
    computed from the 16 kHz waveform so RMS/band/peak values are directly
    comparable in dBFS.
    """
    if input_key != "waveform":
        raise ValueError(f"Level features require waveform input, got {input_key!r}.")
    return waveform_level_features(x)

## 4. Text encoder yang di-fine-tune

`TextEncoderWrapper` bawaan proyek membekukan encoder teks (`@torch.no_grad` + cache), sehingga
tidak bisa dipakai untuk fine-tune. Cabang kecil di bawah ini **meneruskan gradien** ke IndoBERT.

In [5]:
# ----------------------------------------------------------------------------- Cell 5: trainable text branch


class TrainableIndoBertTextBranch(nn.Module):
    """Fine-tunable IndoBERT producing bounded balance targets r (B, 4)."""

    is_pretrained = True

    def __init__(self, model_name: str, hidden_dim: int = 128, dropout: float = 0.15, max_length: int = 64):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        embed_dim = int(self.model.config.hidden_size)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 4),
        )

    def forward(self, prompts, device) -> torch.Tensor:
        encoded = self.tokenizer(
            list(prompts), padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        hidden = self.model(**encoded).last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return 6.0 * torch.tanh(self.head(pooled) / 6.0)

## 5. Audio encoder registry

Antarmuka seragam: `forward(input) -> tokens (B, 4, T', D)`.
- `cnn_scratch` → CNN 1D scratch pada waveform 16 kHz, menghasilkan 25 token temporal.
- `ast` / `mert` / `beats` → encoder pretrained dari waveform (S = 128000), dijalankan per-stem
  dengan bobot yang sama (shared), semua parameter **trainable**.
- Semua encoder memakai input waveform 16 kHz dan level features dari waveform yang sama.

In [6]:
# ----------------------------------------------------------------------------- Cell 6: audio encoders
BEATS_URL = "https://huggingface.co/lpepino/beats_ckpts/resolve/main/BEATs_iter3_plus_AS2M.pt"


def download_beats_checkpoint(path: Path) -> Path:
    path = Path(path)
    if path.is_file():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading BEATs checkpoint -> {path}")
    from urllib.request import urlretrieve

    urlretrieve(BEATS_URL, path)
    return path


def set_encoder_dropout(encoder: nn.Module | None, p: float) -> None:
    """Set dropout probability only inside the audio encoder.

    Text-branch dropout is controlled separately by CONFIG['dropout_text'].
    """
    if encoder is None:
        return
    p = float(p)
    for module in encoder.modules():
        if isinstance(module, nn.Dropout):
            module.p = p


class ScratchWaveformEncoder(nn.Module):
    """Scratch 1D CNN on raw 16 kHz waveform, returning 25 temporal tokens per stem."""

    input_key = "waveform"
    is_pretrained = False

    def __init__(self, embed_dim: int = 128, n_tokens: int = 25):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_tokens = n_tokens
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=4, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=4, padding=2),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, stride=4, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        self.dropout = nn.Dropout(0.0)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, n_samples = waveform.shape
        x = waveform.reshape(bsz * n_stems, 1, n_samples)
        h = self.conv(x)
        h = F.adaptive_avg_pool1d(h, self.n_tokens)
        h = h.transpose(1, 2)
        h = self.proj(h)
        h = self.dropout(h)
        return h.reshape(bsz, n_stems, self.n_tokens, self.embed_dim)


class _HFWaveformEncoder(nn.Module):
    input_key = "waveform"
    is_pretrained = True
    sample_rate = 16000

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:  # (N, S) -> (N, T, D)
        raise NotImplementedError

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        bsz, n_stems, _ = waveform.shape
        flat = waveform.reshape(bsz * n_stems, waveform.shape[-1])
        hidden = self._encode_waveform(flat)
        return hidden.reshape(bsz, n_stems, hidden.shape[-2], hidden.shape[-1])


class ASTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel

        self.extractor = ASTFeatureExtractor.from_pretrained(model_name)
        self.model = ASTModel.from_pretrained(model_name)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        return self.model(feats["input_values"].to(waveforms.device)).last_hidden_state


class MERTEncoder(_HFWaveformEncoder):
    def __init__(self, model_name: str = "m-a-p/MERT-v1-95M"):
        super().__init__()
        from transformers import AutoModel, Wav2Vec2FeatureExtractor

        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name, trust_remote_code=True)
        self.embed_dim = int(self.model.config.hidden_size)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        feats = self.extractor(
            list(waveforms.detach().cpu().numpy()), sampling_rate=self.sample_rate, return_tensors="pt"
        )
        outputs = self.model(feats["input_values"].to(waveforms.device))
        return outputs.last_hidden_state


class BEATsEncoder(_HFWaveformEncoder):
    def __init__(self, checkpoint: str | None = None):
        super().__init__()
        try:
            from beats.BEATs import BEATs, BEATsConfig
        except Exception as exc:  # pragma: no cover - dependency gate
            raise RuntimeError(
                "BEATs requires Microsoft's `beats` package: "
                "`pip install git+https://github.com/microsoft/unilm.git#subdirectory=beats`. "
                f"Original error: {exc}"
            ) from exc

        ckpt = download_beats_checkpoint(Path(checkpoint) if checkpoint else Path.home() / "beats_data" / "BEATs_iter3_plus_AS2M.pt")
        payload = torch.load(ckpt, map_location="cpu")
        config = BEATsConfig(payload["cfg"])
        self.model = BEATs(config)
        self.model.load_state_dict(payload["model"])
        self.embed_dim = int(config.encoder_embed_dim)

    def _encode_waveform(self, waveforms: torch.Tensor) -> torch.Tensor:
        out = self.model.extract_features(waveforms)
        return out[0] if isinstance(out, tuple) else out


ENCODER_REGISTRY = {
    "cnn_scratch": ScratchWaveformEncoder,
    "ast": ASTEncoder,
    "mert": MERTEncoder,
    "beats": BEATsEncoder,
}


def build_encoder(name: str) -> nn.Module:
    if name not in ENCODER_REGISTRY:
        raise ValueError(f"Unknown encoder '{name}'. Available: {sorted(ENCODER_REGISTRY)}")
    return ENCODER_REGISTRY[name]()


## 6. `EncoderProbeModel` (audio encoder + IndoBERT + head M3)

Hanya blok audio encoder yang berganti. Head M3, rumus `g`, dan kontrak loss tetap identik.
`audio_input_key` sekarang selalu `waveform`, sehingga semua encoder dievaluasi dengan
`evaluate_model` yang sama.

In [ ]:
# ----------------------------------------------------------------------------- Cell 7: model


class EncoderProbeModel(nn.Module):
    """M3 structured two-branch head with a swappable audio encoder and a fine-tuned IndoBERT branch."""

    def __init__(self, encoder_name: str, mode: str, config: dict):
        super().__init__()
        if mode not in ("bimodal", "audio_only", "text_only"):
            raise ValueError(f"Invalid mode: {mode}")
        self.mode = mode
        self.encoder_name = encoder_name
        # text_only tidak memakai audio encoder (g = r): jangan bangun/unduh encoder berat.
        self.encoder = None if mode == "text_only" else build_encoder(encoder_name)
        self.audio_input_key = self.encoder.input_key if self.encoder is not None else "audio"

        d = config["d_model"]
        self.enc_proj = nn.Linear(self.encoder.embed_dim if self.encoder is not None else d, d)
        self.stem_embedding = nn.Parameter(torch.zeros(1, 4, 1, d))
        self.level_mlp = nn.Sequential(nn.Linear(N_LEVEL_FEATS, d), nn.GELU(), nn.Linear(d, d))
        self.stem_norm = nn.LayerNorm(d)
        self.loudness_head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
        self.text_branch = TrainableIndoBertTextBranch(
            config["text_model"], config["text_hidden"], config["dropout_text"]
        )

    def forward(self, audio_input, prompts, active_mask=None, clamp_output: bool = False) -> dict:
        device = audio_input.device
        bsz = audio_input.shape[0]
        if active_mask is not None:
            mask = active_mask.to(device=device, dtype=torch.float32)
        else:
            mask = torch.ones((bsz, 4), device=device)
        count = mask.sum(dim=1, keepdim=True).clamp_min(1.0)

        l_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "text_only":
            level = compute_level_features(audio_input, self.audio_input_key)
            tokens = self.encoder(audio_input)
            h = self.enc_proj(tokens) + self.stem_embedding + self.level_mlp(level).unsqueeze(2)
            l_pred = self.loudness_head(self.stem_norm(h.mean(dim=2))).squeeze(-1)
        l_ref = (l_pred * mask).sum(dim=1, keepdim=True) / count

        r_pred = torch.zeros((bsz, 4), device=device)
        if self.mode != "audio_only":
            raw_r = self.text_branch(prompts, device)
            r_pred = (raw_r - (raw_r * mask).sum(dim=1, keepdim=True) / count) * mask

        g_pred = (l_ref + r_pred - l_pred) * mask
        if clamp_output or not self.training:
            g_pred = g_pred.clamp(-18.0, 12.0)
        return {"gains_pred": g_pred, "loudness_pred": l_pred, "balance_pred": r_pred}

    def build_param_groups(self, config: dict) -> list[dict]:
        """Separate encoder / text / head groups so tuning does not leak across branches."""
        groups: list[dict] = []
        if self.encoder is not None:
            enc_lr = config["lr_audio"] if getattr(self.encoder, "is_pretrained", False) else config["lr_scratch"]
            groups.append({
                "params": list(self.encoder.parameters()),
                "lr": enc_lr,
                "weight_decay": config["wd_audio"],
            })

        head_modules = [self.enc_proj, self.level_mlp, self.stem_norm, self.loudness_head, self.text_branch.head]
        head_params = [p for module in head_modules for p in module.parameters()] + [self.stem_embedding]
        groups.append({
            "params": list(self.text_branch.model.parameters()),
            "lr": config["lr_text"],
            "weight_decay": config["wd_text"],
        })
        groups.append({
            "params": head_params,
            "lr": config["lr_head"],
            "weight_decay": config["wd_head"],
        })
        return groups


## 7. Train / eval loop

Reuse `compute_multimodal_loss` dan `evaluate_model`. Early stopping pada val MAE, AMP, gradient
accumulation, cosine schedule. Bobot aux loss mengikuti mode ablasi.

In [ ]:
# ----------------------------------------------------------------------------- Cell 8: training loop


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def env_info() -> dict:
    info = {"torch": torch.__version__, "device": str(DEVICE)}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["cuda"] = torch.version.cuda
    return info


def run_one(
    encoder_name: str,
    mode: str,
    seed: int,
    config: dict,
    train_ds: Dataset,
    val_ds: Dataset,
    device=DEVICE,
    fold_id: int | None = None,
    params: dict | None = None,
    eval_test: bool = False,
    save_state: bool = False,
    checkpoint_path: Path | None = None,
) -> dict:
    seed_everything(seed)
    model = EncoderProbeModel(encoder_name, mode, config).to(device)
    set_encoder_dropout(model.encoder, config["dropout_audio"])
    groups = model.build_param_groups(config)
    optimizer = torch.optim.AdamW(groups, weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(config["epochs"], 1), eta_min=config["eta_min"]
    )
    amp = bool(config["use_amp"] and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    all_params = [p for g in groups for p in g["params"]]

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=NUM_WORKERS)
    accum = max(1, int(config["grad_accum_steps"]))
    lam_l, lam_r = AUX_LAMBDA[mode]

    best = {"val_mae": float("inf"), "epoch": 0, "state": None, "metrics": {}}
    history: list[dict] = []
    t0 = time.time()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            audio = batch[model.audio_input_key].to(device=device, dtype=torch.float32)
            mask = batch["active_mask"].to(device=device)
            with torch.amp.autocast(device_type=device.type, enabled=amp):
                out = model(audio, batch["prompt"], active_mask=mask, clamp_output=False)
                losses = compute_multimodal_loss(
                    out,
                    batch["gains_db"].to(device=device, dtype=torch.float32),
                    batch["loudness_db"].to(device=device, dtype=torch.float32),
                    batch["balance_target"].to(device=device, dtype=torch.float32),
                    active_mask=mask,
                    lambda_loudness=lam_l,
                    lambda_balance=lam_r,
                )
                loss = losses["loss"]
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                if config["grad_clip"] > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(all_params, config["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        val = evaluate_model(model, val_loader, device=device)
        history.append({
            "epoch": epoch,
            "val_mae_db": val["mae_db"],
            "val_rmse_db": val["rmse_db"],
            "val_band_mae_db": val["band_mae_db"],
            "val_success_1db": val["success_rate_1db"],
        })
        fold_tag = f"/f{fold_id}" if fold_id is not None else ""
        print(
            f"[{encoder_name:11s}/{mode:10s}{fold_tag}/seed{seed}] ep{epoch:02d} "
            f"val_MAE={val['mae_db']:.4f} RMSE={val['rmse_db']:.4f} band={val['band_mae_db']:.4f}"
        )
        if val["mae_db"] < best["val_mae"] - 1e-4:
            state = copy.deepcopy(model.state_dict()) if save_state else None
            best = {"val_mae": val["mae_db"], "epoch": epoch, "state": state, "metrics": val}
        elif config["patience"] and (epoch - best["epoch"]) >= config["patience"]:
            print(f"  early stop at epoch {epoch} (best {best['epoch']})")
            break

    if save_state and best["state"] is not None:
        model.load_state_dict(best["state"])

    test_human = None
    test_synthetic = None
    if eval_test:
        if Path(config["test_human_manifest"]).is_file():
            test_loader = DataLoader(
                EncoderDataset(config["test_human_manifest"]),
                batch_size=config["batch_size"],
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            test_human = evaluate_model(model, test_loader, device=device)
        if Path(config["test_synthetic_manifest"]).is_file():
            test_loader = DataLoader(
                EncoderDataset(config["test_synthetic_manifest"]),
                batch_size=config["batch_size"],
                shuffle=False,
                num_workers=NUM_WORKERS,
            )
            test_synthetic = evaluate_model(model, test_loader, device=device)

    if checkpoint_path is not None and save_state:
        checkpoint_path = Path(checkpoint_path)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "state_dict": model.state_dict(),
            "config": config,
            "params": params or {},
            "encoder": encoder_name,
            "mode": mode,
            "seed": seed,
            "best_epoch": best["epoch"],
            "validation": best["metrics"],
            "test_human": test_human,
            "test_synthetic": test_synthetic,
        }, checkpoint_path)

    return {
        "encoder": encoder_name,
        "mode": mode,
        "seed": seed,
        "fold": fold_id,
        "params": params or {},
        "member": MEMBER,
        "env": env_info(),
        "trainable_params": n_params,
        "train_sec": round(time.time() - t0, 1),
        "best_epoch": best["epoch"],
        "validation": best["metrics"],
        "test_human": test_human,
        "test_synthetic": test_synthetic,
        "checkpoint": str(checkpoint_path) if checkpoint_path is not None and save_state else None,
        "history": history,
    }


## 8. 5-fold CV + Optuna

Cell 9 menjalankan stage yang dipilih di Cell 1.

### Audio tuning per member
- 5-fold song-level pada 80 lagu train.
- Setiap fold: 64 lagu train / 16 lagu validation.
- 1 seed (`42`).
- Optuna TPE, objective = mean validation MAE across 5 folds.
- Search space: `lr_audio`/`lr_scratch`, `wd_audio`, `dropout_audio`.
- Output: `best_audio_config_<encoder>.json`, `cv_summary_<encoder>.csv`, `optuna/*.db`.

### Text tuning global
- Hanya dijalankan sekali oleh member pemilik encoder pemenang.
- Memakai best audio config encoder tersebut.
- Search space: `lr_text`, `dropout_text`, `text_hidden`.
- Output: `outputs/encoder_selection/text/text_config.json`.

### Final model
- Best audio config + best text config.
- Train 80 lagu, early stop di 20 lagu validation.
- 3 seed: `42, 123, 2026`.
- Evaluasi `test_human` dan `test_synthetic`.
- Output: `final/final_<encoder>_seed*.json` dan `best_model_<encoder>_seed*.pt`.

Test set tidak pernah dipakai selama CV atau tuning.


In [ ]:
# ----------------------------------------------------------------------------- Cell 9: 5-fold CV + Optuna + final
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
    OPTUNA_AVAILABLE = True
except Exception as exc:  # pragma: no cover
    optuna = None
    OPTUNA_AVAILABLE = False
    print("Optuna belum tersedia:", exc)


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def compact_result(result: dict) -> dict:
    out = {k: v for k, v in result.items() if k not in ("history", "validation", "test_human", "test_synthetic")}
    val = dict(result.get("validation", {}))
    val.pop("per_song", None)
    out["validation"] = val
    for key in ("test_human", "test_synthetic"):
        if result.get(key):
            m = dict(result[key])
            m.pop("per_song", None)
            out[key] = m
    return out


def make_config(params: dict) -> dict:
    cfg = copy.deepcopy(CONFIG)
    cfg.update(params or {})
    return cfg


def mean_std(values):
    arr = np.asarray(list(values), dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))


def run_one_with_retry(encoder_name, mode, seed, config, train_ds, val_ds, **kwargs):
    try:
        return run_one(encoder_name, mode, seed, config, train_ds, val_ds, **kwargs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        cfg = copy.deepcopy(config)
        cfg["batch_size"] = 2
        cfg["grad_accum_steps"] = 8
        print("OOM -> retry dengan batch_size=2, grad_accum_steps=8 (effective batch tetap 16)")
        return run_one(encoder_name, mode, seed, cfg, train_ds, val_ds, **kwargs)


def fold_datasets(fold: dict):
    k = int(fold["fold"])
    train_ds = EncoderDataset(FOLD_DIR / f"fold{k}_train.json")
    val_ds = EncoderDataset(FOLD_DIR / f"fold{k}_val.json")
    return train_ds, val_ds


def evaluate_params_cv(encoder_name: str, params: dict, tag: str, mode: str = "bimodal",
                       seed: int | None = None, folds: list[dict] | None = None) -> list[dict]:
    seed = int(CONFIG["tuning_seed"]) if seed is None else int(seed)
    folds = FOLDS if folds is None else folds
    metrics = []
    for fold in folds:
        train_ds, val_ds = fold_datasets(fold)
        cfg = make_config(params)
        result = run_one_with_retry(
            encoder_name,
            mode,
            seed,
            cfg,
            train_ds,
            val_ds,
            fold_id=int(fold["fold"]),
            params=params,
            eval_test=False,
            save_state=False,
        )
        save_json(CV_DIR / f"{tag}_fold{fold['fold']}.json", compact_result(result))
        row = {"fold": int(fold["fold"])}
        val = dict(result["validation"])
        val.pop("per_song", None)
        row.update(val)
        metrics.append(row)
    return metrics


def audio_space(encoder_name: str) -> dict:
    return AUDIO_SEARCH_SPACE["scratch" if encoder_name == "cnn_scratch" else "pretrained"]


def suggest_params(trial, space: dict) -> dict:
    return {key: trial.suggest_categorical(key, values) for key, values in space.items()}


def make_audio_objective(encoder_name: str):
    space = audio_space(encoder_name)

    def objective(trial):
        params = suggest_params(trial, space)
        tag = f"audio_{encoder_name}_trial{trial.number:03d}"
        metrics = evaluate_params_cv(encoder_name, params, tag=tag)
        maes = [m["mae_db"] for m in metrics]
        trial.set_user_attr("params", params)
        trial.set_user_attr("fold_metrics", metrics)
        for i, _ in enumerate(metrics):
            trial.report(float(np.mean(maes[: i + 1])), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(maes))

    return objective


def save_study_summary(study, path):
    import csv

    rows = []
    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue
        params = trial.user_attrs.get("params", trial.params)
        fold_metrics = trial.user_attrs.get("fold_metrics", [])
        if not fold_metrics:
            continue
        maes = [m["mae_db"] for m in fold_metrics]
        rmses = [m["rmse_db"] for m in fold_metrics]
        mae_mean, mae_std = mean_std(maes)
        rmse_mean, _ = mean_std(rmses)
        rows.append({
            "trial": trial.number,
            "params": json.dumps(params, sort_keys=True),
            "cv_mae_mean": round(mae_mean, 4),
            "cv_mae_std": round(mae_std, 4),
            "cv_rmse_mean": round(rmse_mean, 4),
        })
    rows = sorted(rows, key=lambda r: r["cv_mae_mean"])
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if rows:
        with path.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
    return rows


def run_audio_tuning():
    if not OPTUNA_AVAILABLE:
        raise RuntimeError("Optuna belum terpasang. Jalankan ulang sel Setup.")
    if "bimodal" not in SHARDS[MEMBER]["modes"]:
        print("member ini tidak punya mode bimodal; skip audio tuning")
        return

    t_start = time.time()
    for encoder_name in SHARDS[MEMBER]["encoders"]:
        study_path = STUDY_DIR / f"audio_{encoder_name}.db"
        study = optuna.create_study(
            study_name=f"audio_{encoder_name}_member{MEMBER}",
            direction="minimize",
            storage=f"sqlite:///{study_path}",
            load_if_exists=True,
            sampler=TPESampler(seed=int(CONFIG["tuning_seed"])),
            pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
        )
        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        remaining = max(0, int(CONFIG["n_trials"]) - len(completed))
        if MAX_TRIALS_PER_SESSION and MAX_TRIALS_PER_SESSION > 0:
            remaining = min(remaining, MAX_TRIALS_PER_SESSION)
        if remaining:
            timeout = max(1, int(TIME_BUDGET_MIN * 60 - (time.time() - t_start)))
            study.optimize(make_audio_objective(encoder_name), n_trials=remaining, timeout=timeout)
        completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not completed:
            print(f"{encoder_name}: belum ada trial selesai")
            continue
        best = study.best_trial
        params = best.user_attrs.get("params", best.params)
        fold_metrics = best.user_attrs.get("fold_metrics", [])
        if not fold_metrics:
            print(f"{encoder_name}: best trial tidak punya fold_metrics; skip")
            continue
        maes = [m["mae_db"] for m in fold_metrics]
        rmses = [m["rmse_db"] for m in fold_metrics]
        mae_mean, mae_std = mean_std(maes)
        rmse_mean, _ = mean_std(rmses)
        payload = {
            "encoder": encoder_name,
            "member": MEMBER,
            "mode": "bimodal",
            "params": params,
            "cv_mae_mean": round(mae_mean, 4),
            "cv_mae_std": round(mae_std, 4),
            "cv_rmse_mean": round(rmse_mean, 4),
            "n_folds": len(fold_metrics),
            "tuning_seed": int(CONFIG["tuning_seed"]),
            "n_trials": len(completed),
            "fold_metrics": fold_metrics,
        }
        save_json(RUN_DIR / f"best_audio_config_{encoder_name}.json", payload)
        save_study_summary(study, RUN_DIR / f"cv_summary_{encoder_name}.csv")
        print(f"[audio] {encoder_name}: CV MAE {mae_mean:.4f} ± {mae_std:.4f} dB | params={params}")


def load_best_audio(encoder_name: str) -> dict:
    candidates = [RUN_DIR / f"best_audio_config_{encoder_name}.json"]
    candidates.extend(sorted(OUT_DIR.glob(f"member*/best_audio_config_{encoder_name}.json")))
    for path in candidates:
        if Path(path).is_file():
            return json.loads(Path(path).read_text())
    raise FileNotFoundError(f"best_audio_config_{encoder_name}.json belum ada")


def make_text_objective(encoder_name: str, audio_params: dict):
    def objective(trial):
        text_params = suggest_params(trial, TEXT_SEARCH_SPACE)
        params = {**audio_params, **text_params}
        tag = f"text_{encoder_name}_trial{trial.number:03d}"
        metrics = evaluate_params_cv(encoder_name, params, tag=tag)
        maes = [m["mae_db"] for m in metrics]
        trial.set_user_attr("params", text_params)
        trial.set_user_attr("fold_metrics", metrics)
        for i, _ in enumerate(metrics):
            trial.report(float(np.mean(maes[: i + 1])), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(maes))

    return objective


def run_text_tuning():
    if not OPTUNA_AVAILABLE:
        raise RuntimeError("Optuna belum terpasang. Jalankan ulang sel Setup.")
    if TEXT_TUNING_ENCODER is None or TEXT_TUNING_MEMBER is None:
        print("TEXT_TUNING_ENCODER / TEXT_TUNING_MEMBER belum diset; skip text tuning")
        return
    if int(MEMBER) != int(TEXT_TUNING_MEMBER):
        print(f"text tuning hanya dijalankan oleh member {TEXT_TUNING_MEMBER}; member ini {MEMBER}")
        return
    if TEXT_TUNING_ENCODER not in SHARDS[MEMBER]["encoders"]:
        print(f"{TEXT_TUNING_ENCODER} bukan encoder member {MEMBER}; skip")
        return

    best_audio = load_best_audio(TEXT_TUNING_ENCODER)
    audio_params = best_audio["params"]
    study_path = TEXT_DIR / "optuna_text.db"
    study = optuna.create_study(
        study_name=f"text_{TEXT_TUNING_ENCODER}",
        direction="minimize",
        storage=f"sqlite:///{study_path}",
        load_if_exists=True,
        sampler=TPESampler(seed=int(CONFIG["tuning_seed"])),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    )
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    remaining = max(0, int(CONFIG["n_trials"]) - len(completed))
    if MAX_TRIALS_PER_SESSION and MAX_TRIALS_PER_SESSION > 0:
        remaining = min(remaining, MAX_TRIALS_PER_SESSION)
    if remaining:
        timeout = max(1, int(TIME_BUDGET_MIN * 60))
        study.optimize(make_text_objective(TEXT_TUNING_ENCODER, audio_params), n_trials=remaining, timeout=timeout)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not completed:
        print("text tuning: belum ada trial selesai")
        return
    best = study.best_trial
    text_params = best.user_attrs.get("params", best.params)
    fold_metrics = best.user_attrs.get("fold_metrics", [])
    if not fold_metrics:
        print("text tuning: best trial tidak punya fold_metrics; skip")
        return
    maes = [m["mae_db"] for m in fold_metrics]
    rmses = [m["rmse_db"] for m in fold_metrics]
    mae_mean, mae_std = mean_std(maes)
    rmse_mean, _ = mean_std(rmses)
    payload = {
        "encoder": TEXT_TUNING_ENCODER,
        "member": MEMBER,
        "params": text_params,
        "audio_params": audio_params,
        "cv_mae_mean": round(mae_mean, 4),
        "cv_mae_std": round(mae_std, 4),
        "cv_rmse_mean": round(rmse_mean, 4),
        "n_folds": len(fold_metrics),
        "tuning_seed": int(CONFIG["tuning_seed"]),
        "n_trials": len(completed),
        "fold_metrics": fold_metrics,
    }
    save_json(TEXT_DIR / "text_config.json", payload)
    save_study_summary(study, TEXT_DIR / "cv_summary_text.csv")
    print(f"[text] {TEXT_TUNING_ENCODER}: CV MAE {mae_mean:.4f} ± {mae_std:.4f} dB | params={text_params}")


def run_final():
    if FINAL_ENCODER is None:
        print("FINAL_ENCODER belum diset; skip final")
        return
    if FINAL_ENCODER not in SHARDS[MEMBER]["encoders"]:
        print(f"{FINAL_ENCODER} bukan encoder member {MEMBER}; skip final")
        return

    best_audio = load_best_audio(FINAL_ENCODER)
    text_path = TEXT_DIR / "text_config.json"
    if text_path.is_file():
        text_cfg = json.loads(text_path.read_text())
        text_params = text_cfg["params"]
    else:
        text_params = {
            "lr_text": CONFIG["lr_text"],
            "dropout_text": CONFIG["dropout_text"],
            "text_hidden": CONFIG["text_hidden"],
        }
    params = {**best_audio["params"], **text_params}
    cfg = make_config(params)

    train_ds = EncoderDataset(CONFIG["train_manifest"])
    val_ds = EncoderDataset(CONFIG["val_manifest"])
    results = []
    for seed in CONFIG["seeds"]:
        ckpt = FINAL_DIR / f"best_model_{FINAL_ENCODER}_seed{seed}.pt" if SAVE_FINAL_CHECKPOINT else None
        result = run_one_with_retry(
            FINAL_ENCODER,
            "bimodal",
            int(seed),
            cfg,
            train_ds,
            val_ds,
            params=params,
            eval_test=True,
            save_state=True,
            checkpoint_path=ckpt,
        )
        save_json(FINAL_DIR / f"final_{FINAL_ENCODER}_seed{seed}.json", result)
        results.append(result)

    summary = {
        "encoder": FINAL_ENCODER,
        "member": MEMBER,
        "params": params,
        "n_seeds": len(results),
        "seeds": list(CONFIG["seeds"]),
        "test_human": {},
        "test_synthetic": {},
    }
    for key in ("test_human", "test_synthetic"):
        vals = [r[key]["mae_db"] for r in results if r.get(key)]
        rmses = [r[key]["rmse_db"] for r in results if r.get(key)]
        if vals:
            mae_mean, mae_std = mean_std(vals)
            rmse_mean, _ = mean_std(rmses)
            summary[key] = {
                "mae_db_mean": round(mae_mean, 4),
                "mae_db_std": round(mae_std, 4),
                "rmse_db_mean": round(rmse_mean, 4),
            }
    save_json(FINAL_DIR / f"final_summary_{FINAL_ENCODER}.json", summary)
    print(f"[final] {FINAL_ENCODER}: {summary['test_human']} | {summary['test_synthetic']}")


def run_text_only():
    if "text_only" not in SHARDS[MEMBER]["modes"]:
        print(f"text_only bukan mode member {MEMBER}; skip")
        return
    text_path = TEXT_DIR / "text_config.json"
    if text_path.is_file():
        params = json.loads(text_path.read_text())["params"]
    else:
        params = {
            "lr_text": CONFIG["lr_text"],
            "dropout_text": CONFIG["dropout_text"],
            "text_hidden": CONFIG["text_hidden"],
        }
    metrics = evaluate_params_cv("ast", params, tag="text_only", mode="text_only")
    import csv

    path = RUN_DIR / "cv_summary_text_only.csv"
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(metrics[0].keys()))
        writer.writeheader()
        writer.writerows(metrics)
    maes = [m["mae_db"] for m in metrics]
    mae_mean, mae_std = mean_std(maes)
    print(f"[text_only] CV MAE {mae_mean:.4f} ± {mae_std:.4f} dB")


if RUN_AUDIO_TUNING:
    run_audio_tuning()
if RUN_TEXT_TUNING:
    run_text_tuning()
if RUN_FINAL:
    run_final()
if RUN_TEXT_ONLY:
    run_text_only()
if not any([RUN_AUDIO_TUNING, RUN_TEXT_TUNING, RUN_FINAL, RUN_TEXT_ONLY]):
    print("Tidak ada stage yang dijalankan.")
    print("Set RUN_AUDIO_TUNING / RUN_TEXT_TUNING / RUN_FINAL / RUN_TEXT_ONLY di Cell 1.")


## 9. Leaderboard

Leaderboard menggabungkan:
- `best_audio_config_<encoder>.json` → CV MAE mean ± std.
- `final/final_summary_<encoder>.json` → test MAE mean ± std.

Pemenang dipilih dari **CV MAE terendah**, lalu dikonfirmasi pada test set.


In [ ]:
# ----------------------------------------------------------------------------- Cell 10: leaderboard (CV + final)
import csv


def load_json(path):
    return json.loads(Path(path).read_text())


def load_audio_configs():
    rows = []
    for path in sorted(OUT_DIR.glob("member*/best_audio_config_*.json")):
        try:
            d = load_json(path)
        except Exception as exc:  # noqa: BLE001
            print("skip", path, exc)
            continue
        rows.append(d)
    return rows


def load_final_summaries():
    out = {}
    for path in sorted(OUT_DIR.glob("final/final_summary_*.json")):
        try:
            d = load_json(path)
        except Exception as exc:  # noqa: BLE001
            print("skip", path, exc)
            continue
        out[d["encoder"]] = d
    return out


AUDIO_ROWS = load_audio_configs()
FINAL_ROWS = load_final_summaries()

TEXT_CFG = {}
_text_path = TEXT_DIR / "text_config.json"
if _text_path.is_file():
    TEXT_CFG = json.loads(_text_path.read_text()).get("params", {})

SUMMARY = []
for row in AUDIO_ROWS:
    encoder = row["encoder"]
    final = FINAL_ROWS.get(encoder, {})
    th = final.get("test_human", {})
    ts = final.get("test_synthetic", {})
    SUMMARY.append({
        "encoder": encoder,
        "member": row.get("member"),
        "cv_mae_mean": row.get("cv_mae_mean"),
        "cv_mae_std": row.get("cv_mae_std"),
        "cv_rmse_mean": row.get("cv_rmse_mean"),
        "test_human_mae_mean": th.get("mae_db_mean"),
        "test_human_mae_std": th.get("mae_db_std"),
        "test_synthetic_mae_mean": ts.get("mae_db_mean"),
        "test_synthetic_mae_std": ts.get("mae_db_std"),
        "n_seeds": final.get("n_seeds"),
        "params": json.dumps(row.get("params", {}), sort_keys=True),
        "text_params": json.dumps(TEXT_CFG, sort_keys=True),
    })

SUMMARY = sorted(SUMMARY, key=lambda r: (r["cv_mae_mean"] is None, r["cv_mae_mean"] or float("inf")))
if SUMMARY:
    csv_path = OUT_DIR / "audio_encoder_results.csv"
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(SUMMARY[0].keys()))
        writer.writeheader()
        writer.writerows(SUMMARY)
    print("wrote", csv_path)
    print(f"{'encoder':12s} {'member':>6s} {'CV MAE':>16s} {'test_human':>16s} {'test_synth':>16s}")
    for r in SUMMARY:
        cv = f"{r['cv_mae_mean']:.4f}±{r['cv_mae_std']:.4f}" if r["cv_mae_mean"] is not None else "-"
        th = f"{r['test_human_mae_mean']:.4f}" if r["test_human_mae_mean"] is not None else "-"
        ts = f"{r['test_synthetic_mae_mean']:.4f}" if r["test_synthetic_mae_mean"] is not None else "-"
        print(f"{r['encoder']:12s} {str(r['member']):>6s} {cv:>16s} {th:>16s} {ts:>16s}")
    print("\nBest by CV MAE:", SUMMARY[0]["encoder"], SUMMARY[0]["cv_mae_mean"])
else:
    print("Belum ada best_audio_config_*.json. Jalankan audio tuning dulu.")


In [11]:
# ----------------------------------------------------------------------------- Cell 11: figure (CV MAE by encoder)
if SUMMARY:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    labels = [r["encoder"] for r in SUMMARY]
    vals = [r["cv_mae_mean"] if r["cv_mae_mean"] is not None else 0.0 for r in SUMMARY]
    errs = [r["cv_mae_std"] if r["cv_mae_std"] is not None else 0.0 for r in SUMMARY]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, vals, yerr=errs, capsize=3)
    ax.set_ylabel("5-fold CV MAE (dB)")
    ax.set_title("Audio encoder comparison (Optuna, 5-fold CV)")
    fig.tight_layout()
    fig_path = OUT_DIR / "audio_encoder_mae.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.close(fig)
else:
    print("nothing to plot yet")


nothing to plot yet


## 10. Parity / sanity checks

Memastikan:
- 5 fold song-level, 80 lagu train, 16 lagu validation per fold.
- Tidak ada lagu test di fold.
- Validation 20 lagu dan test 50 lagu tidak masuk CV.
- Semua parameter trainable.
- Text config tunggal dan sama untuk semua member.
- Shard encoder menutup semua kombinasi.


In [ ]:
# ----------------------------------------------------------------------------- Cell 12: parity checks
import collections


def _manifest_records(path):
    data = json.loads(Path(path).read_text())
    return data["samples"] if isinstance(data, dict) and "samples" in data else data

FIXED_KEYS = [
    "text_model", "d_model", "text_hidden", "dropout_text", "dropout_audio",
    "batch_size", "grad_accum_steps", "epochs", "patience",
    "lr_head", "lr_audio", "lr_scratch", "lr_text",
    "wd_audio", "wd_text", "wd_head", "eta_min", "grad_clip", "seeds",
    "n_folds", "cv_seed", "tuning_seed",
]
for key in FIXED_KEYS:
    assert key in CONFIG, f"missing fixed knob: {key}"
assert set(CONFIG["modes"]) == {"bimodal", "audio_only", "text_only"}
assert CONFIG["batch_size"] * CONFIG["grad_accum_steps"] == effective_batch, "effective batch harus 16"
assert MEMBER in SHARDS

# Shard coverage.
coverage = collections.defaultdict(set)
for member, shard in SHARDS.items():
    for enc in shard["encoders"]:
        for mode in shard["modes"]:
            coverage[(enc, mode)].add(member)
expected = {(e, m) for e in CONFIG["encoders"] for m in ("bimodal", "audio_only")}
missing = expected - set(coverage)
assert not missing, f"shard tidak menutup: {sorted(missing)}"
text_only_members = [m for m, s in SHARDS.items() if "text_only" in s["modes"]]
assert len(text_only_members) == 1, f"text_only harus tepat satu anggota, bukan {text_only_members}"

# Fold checks: 80 train songs, 5 folds, 16 val songs, disjoint, no test songs.
fold_payload = globals().get("FOLD_PAYLOAD")
if fold_payload is None:
    assert FOLD_FILE.is_file(), f"fold file belum ada: {FOLD_FILE}"
    fold_payload = json.loads(FOLD_FILE.read_text())
folds = fold_payload["folds"]
assert len(folds) == 5, f"harus 5 fold, bukan {len(folds)}"
all_train_songs, all_val_songs = set(), []
for fold in folds:
    assert len(fold["val_songs"]) == 16, f"fold {fold['fold']} val bukan 16 lagu"
    assert len(fold["train_songs"]) == 64, f"fold {fold['fold']} train bukan 64 lagu"
    assert not (set(fold["train_songs"]) & set(fold["val_songs"])), "train/val overlap"
    all_train_songs.update(fold["train_songs"])
    all_val_songs.extend(fold["val_songs"])
assert len(all_train_songs) == 80, f"lagu train harus 80, bukan {len(all_train_songs)}"
assert len(all_val_songs) == 80 and len(set(all_val_songs)) == 80, "val songs harus cover 80 lagu sekali"
assert set(all_val_songs) == all_train_songs, "val songs harus sama dengan himpunan lagu train"

test_songs = set()
for manifest in (CONFIG["test_human_manifest"], CONFIG["test_synthetic_manifest"]):
    if Path(manifest).is_file():
        test_songs.update(str(r["song"]) for r in _manifest_records(manifest))
assert not (all_train_songs & test_songs), "test song masuk fold CV"

# Nothing frozen.
probe = EncoderProbeModel("cnn_scratch", "bimodal", CONFIG)
frozen = [name for name, p in probe.named_parameters() if not p.requires_grad]
assert not frozen, f"frozen params found: {frozen}"
assert all(p.requires_grad for p in probe.text_branch.model.parameters())
assert EncoderProbeModel("ast", "text_only", CONFIG).encoder is None

# Text config, if present, must be a single global file.
text_cfg = TEXT_DIR / "text_config.json"
if text_cfg.is_file():
    payload = json.loads(text_cfg.read_text())
    assert set(payload["params"]) == set(TEXT_SEARCH_SPACE), f"text params mismatch: {payload['params']}"

print(f"parity OK | 5-fold song-level | 80 train songs | text_only member {text_only_members[0]}")
if text_cfg.is_file():
    print("text config:", json.loads(text_cfg.read_text())["params"])
print("parity checks passed.")


In [ ]:
# ----------------------------------------------------------------------------- Cell 13: package outputs (Kaggle)
import shutil

member_dir = OUT_DIR / f"member{MEMBER}"
member_files = sorted(member_dir.glob("**/*")) if member_dir.is_dir() else []
if member_files:
    archive_base = Path("/kaggle/working") / f"encoder_selection_member{MEMBER}"
    archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUT_DIR))
    print("wrote", archive, f"({archive.stat().st_size / 1e6:.2f} MB)")
    for path in member_files:
        if path.is_file():
            print(" -", path.relative_to(OUT_DIR))
    print("\nUnduh zip dari panel Output, lalu commit JSON/CSV ke outputs/encoder_selection/ "
          f"member{MEMBER}/ di repo.")
else:
    print("belum ada hasil untuk di-zip; jalankan stage tuning terlebih dahulu")


## 11. Kesimpulan & handoff ke step 2

1. Pilih encoder pemenang dari **CV MAE** (mean ± std 5 fold).
2. Jalankan text tuning global pada encoder pemenang.
3. Latih final model 3 seed dan laporkan **test MAE mean ± std**.
4. Bawa encoder + best audio config + best text config ke **step 2** untuk sweep fusion.

**Keterbatasan yang harus ditulis:**
- MERT diberi input 16 kHz (aslinya 24 kHz) — domain shift.
- Pretraining dataset AST/MERT/BEATs berbeda — perbandingan pretraining tidak netral.
- Jumlah parameter encoder berbeda; yang disamakan adalah protokol.
- CV score setelah tuning bersifat optimistis; test set adalah estimasi final.


## Lanjut setelah sesi Kaggle

1. Unduh `encoder_selection_member<N>.zip` dari panel **Output**.
2. Ekstrak ke repo lokal: `outputs/encoder_selection/member<N>/`.
3. Commit + push.
4. Ketua menjalankan **Cell 10 (Leaderboard)**.
5. Setelah audio winner dipilih, jalankan text tuning global di member pemenang.
6. Setelah text tuning, jalankan final model 3 seed.
7. Commit `outputs/encoder_selection/text/` dan `outputs/encoder_selection/final/`.

**Catatan teknis**
- Cache log-mel 593 MB di-upload sebagai dataset; cache waveform 1,5 GB dibangun sekali per sesi.
- `MEMBER=4` meng-install paket BEATs dari GitHub (butuh Internet ON).
- MERT memakai `trust_remote_code=True`.
- Notebook ini hanya memakai satu GPU (`cuda:0`) meskipun accelerator `T4 x2`.
- Optuna study disimpan sebagai SQLite di `member<N>/optuna/`; gunakan `load_if_exists=True`
  agar sesi berikutnya melanjutkan trial yang sudah selesai.
